# Stage 13 — frozen GLM prompt-held-out classifier

Fit one fixed CPU classifier recipe on five whole-prompt training folds, then collect held-out probabilities for the corresponding validation folds. The canonical folds were used in prior experiments, so this is exploratory development evidence. The job does not balance predictions, reconstruct S07 or make a competition submission.


## Private CPU run inputs

| Input | Required version |
|---|---:|
| Name That LLM competition | 170484 |
| Own Stage 06C feature-cache notebook output | 359405443 |

Verify the two exact bindings, accelerator None, Internet off and private visibility in the saved Kaggle editor before running. The first step of the runner repeats the frozen Stage 13A input and fold checks.


In [ ]:
from pathlib import Path
import hashlib
import json
import math
import subprocess
import sys

_admission_source = '"""Stage13A: read-only admission of frozen-base GLM features, never OOF scoring.\n\nRun on CPU with Internet off. Only aggregate evidence is written. No fitting,\nprediction, Torch/custom-code import, object deserialization or balancing occurs.\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom dataclasses import dataclass\nimport hashlib\nimport json\nfrom pathlib import Path\nimport platform\nimport time\nimport zipfile\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn import __version__ as sklearn_version\nfrom sklearn.model_selection import GroupKFold\n\n\nTRAIN_SHA256 = "40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707"\nGLM_SHA256 = "2b41585a9482cef825962a8cc096697933c0b1237e8ede0712d1f8ab361d9414"\nORDERED_TEXT_SHA256 = "6496071874da8f911fafebd9eec6191e56134a97f00226dd8c78805d31f07e36"\nFOLD_SOURCE_SHA256 = "5c27fcdffde2bcfd512d20fb4edec9bd7133430e79ccd71e31875151d235a23d"\nFOLD_MAP_SHA256 = "a55b035686cb7b31f6fbe852ac990668709aec4b913c5d0928d89d843bdbfe62"\nCLASSES = (\n    "claude-fable-5", "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5",\n    "claude-sonnet-5", "gemini-3.1-pro-preview", "gemini-3.6-flash",\n    "gemini-3.7-flash", "gemini-3.8-flash", "glm-5.3", "glm-5.3-flash",\n    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra",\n    "gpt-6-astra", "kimi-k3", "qwen3.8-max",\n)\nCACHE_KEYS = {"features", "ids", "layers", "text_sha256", "token_lengths", "train_sha256"}\n\n\n@dataclass(frozen=True)\nclass Contract:\n    """The CLI always uses PRODUCTION; small contracts exist for fixtures only."""\n    train_sha256: str\n    glm_sha256: str\n    ordered_text_sha256: str\n    fold_map_sha256: str\n    rows: int = 2699\n    prompts: int = 150\n    hidden: int = 4096\n    classes: tuple[str, ...] = CLASSES\n    layers: tuple[int, ...] = (8, 3, 24)\n    folds: int = 5\n\n\nPRODUCTION = Contract(TRAIN_SHA256, GLM_SHA256, ORDERED_TEXT_SHA256, FOLD_MAP_SHA256)\n\n\nclass AdmissionFailure(Exception):\n    """Only a fixed, sanitized check code may leave the audit."""\n\n\ndef check(condition, code):\n    if not bool(condition):\n        raise AdmissionFailure(code)\n\n\ndef json_hash(value):\n    data = json.dumps(value, sort_keys=True, separators=(",", ":")).encode("utf-8")\n    return hashlib.sha256(data).hexdigest()\n\n\ndef sha256(path):\n    result = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            result.update(block)\n    return result.hexdigest()\n\n\ndef unique_file(root, name):\n    matches = [path for path in root.rglob(name) if path.is_file()]\n    check(len(matches) == 1, "unique_" + name.replace(".", "_"))\n    return matches[0]\n\n\ndef npz_headers(path, contract):\n    """Inspect object/oversize headers before NumPy can allocate or deserialize."""\n    expected_names = {key + ".npy" for key in CACHE_KEYS}\n    result = {}\n    with zipfile.ZipFile(path) as archive:\n        names = archive.namelist()\n        check(len(names) == len(set(names)) and set(names) == expected_names, "glm_cache_schema")\n        max_size = contract.rows * len(contract.layers) * contract.hidden * 4 + 2_000_000\n        check(sum(info.file_size for info in archive.infolist()) <= max_size, "glm_cache_size_bound")\n        for name in names:\n            with archive.open(name) as stream:\n                version = np.lib.format.read_magic(stream)\n                check(version in ((1, 0), (2, 0)), "npy_header_version")\n                reader = (np.lib.format.read_array_header_1_0 if version == (1, 0)\n                          else np.lib.format.read_array_header_2_0)\n                shape, fortran, dtype = reader(stream)\n            check(not dtype.hasobject, "npz_object_dtype_rejected")\n            result[name[:-4]] = (shape, dtype)\n    check(result["features"] == ((contract.rows, len(contract.layers), contract.hidden), np.dtype("float32")),\n          "glm_feature_shape_dtype")\n    for name in ("ids", "text_sha256"):\n        shape, dtype = result[name]\n        check(shape == (contract.rows,) and dtype.kind == "U" and dtype.itemsize <= 512,\n              "glm_" + name + "_header")\n    check(result["layers"] == ((len(contract.layers),), np.dtype("int64")), "glm_layers_header")\n    check(result["token_lengths"] == ((contract.rows,), np.dtype("int64")), "glm_token_lengths_header")\n    shape, dtype = result["train_sha256"]\n    check(shape == () and dtype.kind == "U" and dtype.itemsize <= 512, "glm_train_digest_header")\n    return result\n\n\ndef validate_fold_partition(groups, folds, n_folds):\n    """No prompt may cross a future fit/scoring partition; no model is fitted."""\n    groups, folds = np.asarray(groups), np.asarray(folds)\n    check(folds.shape == groups.shape and folds.dtype.kind in "iu", "fold_shape_dtype")\n    check(set(folds.tolist()) == set(range(1, n_folds + 1)), "fold_coverage")\n    mapping = {}\n    for group, fold in zip(groups, folds):\n        key = str(group)\n        check(key not in mapping or mapping[key] == int(fold), "prompt_crosses_folds")\n        mapping[key] = int(fold)\n    summaries = []\n    for number in range(1, n_folds + 1):\n        valid = folds == number\n        fit = ~valid\n        fit_groups, valid_groups = set(groups[fit]), set(groups[valid])\n        check(valid.any() and fit.any() and not (fit_groups & valid_groups), "fold_prompt_exclusion")\n        summaries.append({"fold": number, "fit_rows": int(fit.sum()), "valid_rows": int(valid.sum()),\n                          "fit_prompts": len(fit_groups), "valid_prompts": len(valid_groups),\n                          "prompt_overlap": 0})\n    return mapping, summaries\n\n\ndef audit_inputs(input_root, output_root, *, contract=PRODUCTION):\n    """Return sanitized admission evidence. Existing output directories stay intact."""\n    started = time.monotonic()\n    root, out = Path(input_root), Path(output_root)\n    production = contract == PRODUCTION\n    report = {\n        "schema_version": 1, "status": "FAILED_INPUT_ADMISSION", "contract_is_production": production,\n        "scope": "frozen_base_feature_admission_only", "fit_performed": False,\n        "model_inference_performed": False, "balancing_performed": False,\n        "held_out_predictions_produced": False, "development_reference_admitted": False,\n        "stage13b_batch_coupled_comparison_admitted": False, "full_s07_oof_admitted": False,\n        "eligible_for_future_fold_local_readout_fitting": False,\n        "official_score_estimated": False, "raw_rows_or_predictions_exported": False,\n        "object_deserialization_performed": False, "checks_passed": [],\n        "expected_input_versions": {"competition": 170484, "own_glm_output": 359405443},\n        "input_versions_verified_by_filesystem": False,\n        "runtime_versions": {"python": platform.python_version(), "numpy": np.__version__,\n                             "pandas": pd.__version__, "scikit_learn": sklearn_version},\n        "limits": [\n            "Feature admission is not fitted held-out predictions, a development score or a full-S07 reference.",\n            "Future readouts and learned preprocessing must exclude every scoring-batch prompt.",\n            "Mixed supervised source folds are not admitted for coupled balancing.",\n            "DeBERTa, adapted/full-fit S07 members and other branches are not admitted by this GLM-only audit.",\n            "Recorded cache/extractor provenance is not an independent audit of general pretrained-weight history.",\n            "Immutable input-version bindings require separate saved-UI verification.",\n            "Canonical prompt folds were previously used and remain exploratory development data.",\n        ],\n    }\n    can_write = False\n\n    def passed(code):\n        report["checks_passed"].append(code)\n\n    try:\n        check(not out.exists() or (out.is_dir() and not any(out.iterdir())), "fresh_output_directory")\n        out.mkdir(parents=True, exist_ok=True)\n        can_write = True\n        passed("fresh_output_directory")\n        check(root.is_dir(), "input_root_directory")\n        train_path, cache_path = unique_file(root, "train.csv"), unique_file(root, "glm_features.npz")\n        passed("unique_required_inputs")\n        train_digest, cache_digest = sha256(train_path), sha256(cache_path)\n        check(train_digest == contract.train_sha256, "official_train_sha256")\n        check(cache_digest == contract.glm_sha256, "glm_cache_sha256")\n        report["files"] = {"official_train": {"sha256": train_digest, "bytes": train_path.stat().st_size},\n                           "glm_cache": {"sha256": cache_digest, "bytes": cache_path.stat().st_size}}\n        passed("exact_input_file_hashes")\n        train = pd.read_csv(train_path, dtype=str)\n        check(set(train.columns) == {"id", "text", "prompt_id", "model"}, "official_train_schema")\n        check(len(train) == contract.rows and not train.isna().any().any(), "official_train_rows_nonnull")\n        check(all(train[col].str.len().gt(0).all() for col in train.columns), "official_train_nonempty_strings")\n        check(train.id.is_unique, "official_train_unique_ids")\n        ids = train.id.to_numpy(dtype=str)\n        groups = train.prompt_id.to_numpy(dtype=str)\n        classes = tuple(np.sort(train.model.unique()).tolist())\n        check(classes == contract.classes and len(np.unique(groups)) == contract.prompts, "official_classes_prompts")\n        passed("official_train_axis_contract")\n        text_hashes = np.asarray([hashlib.sha256(value.encode("utf-8")).hexdigest()\n                                  for value in train.text], dtype=str)\n        text_digest = json_hash(text_hashes.tolist())\n        check(text_digest == contract.ordered_text_sha256, "ordered_text_digest")\n        npz_headers(cache_path, contract)\n        passed("numeric_unicode_npz_headers")\n        with np.load(cache_path, allow_pickle=False) as cache:\n            check(np.array_equal(cache["ids"], ids), "glm_exact_official_id_order")\n            check(np.array_equal(cache["text_sha256"], text_hashes), "glm_exact_text_hash_order")\n            check(cache["train_sha256"].item() == contract.train_sha256, "glm_bound_train_digest")\n            check(np.array_equal(cache["layers"], contract.layers), "glm_exact_layer_order")\n            lengths = cache["token_lengths"]\n            check(bool(((lengths > 0) & (lengths <= 512)).all()), "glm_recorded_token_lengths")\n            features = cache["features"]\n            for start in range(0, contract.rows, 256):\n                check(np.isfinite(features[start:start + 256]).all(), "glm_features_finite")\n            del features\n        passed("glm_values_alignment_and_finiteness")\n        labels = {label: i for i, label in enumerate(classes)}\n        y = train.model.map(labels).to_numpy(dtype=int)\n        folds = np.full(contract.rows, -1, dtype=np.int64)\n        dummy = np.zeros((contract.rows, 1), dtype=np.uint8)\n        for number, (_, valid) in enumerate(GroupKFold(contract.folds).split(dummy, y, groups), 1):\n            folds[valid] = number\n        mapping, fold_summary = validate_fold_partition(groups, folds, contract.folds)\n        fold_hash = json_hash(mapping)\n        check(fold_hash == contract.fold_map_sha256, "canonical_prompt_fold_mapping")\n        passed("canonical_future_prompt_fold_partitions")\n        report.update(\n            status=("PASS_FROZEN_BASE_FEATURE_ADMISSION_ONLY" if production\n                    else "PASS_SYNTHETIC_FEATURE_ADMISSION_ONLY"),\n            eligible_for_future_fold_local_readout_fitting=True,\n            rows=contract.rows, prompts=contract.prompts, classes=list(classes),\n            class_supports={label: int((train.model == label).sum()) for label in classes},\n            feature_shape=[contract.rows, len(contract.layers), contract.hidden], feature_dtype="float32",\n            layers=list(contract.layers), classifier_layers=[8, 3], grouping_layer=24,\n            all_feature_values_finite=True, ordered_text_sha256=text_digest,\n            id_axis_sha256=json_hash(ids.tolist()), label_axis_sha256=json_hash(list(classes)),\n            fold_source_path="scripts/stage06c_mixed_confirmation.py", fold_source_sha256=FOLD_SOURCE_SHA256,\n            canonical_prompt_fold_mapping_sha256=fold_hash, fold_partitions=fold_summary,\n            recorded_token_length_min=int(lengths.min()), recorded_token_length_max=int(lengths.max()),\n        )\n    except AdmissionFailure as error:\n        report.update(failure_code=str(error), failure_type="AdmissionFailure")\n    except Exception as error:\n        # Never include library exceptions: they can contain row values or private paths.\n        report.update(failure_code="reader_or_environment_failure", failure_type=type(error).__name__)\n    report["checks_passed_count"] = len(report["checks_passed"])\n    report["seconds"] = round(time.monotonic() - started, 6)\n    report["report_written"] = can_write\n    if can_write:\n        try:\n            (out / "stage13a_input_admission.json").write_text(json.dumps(report, indent=2, sort_keys=True) + "\\n")\n        except Exception as error:\n            report.update(status="FAILED_INPUT_ADMISSION", failure_code="report_write_failure",\n                          failure_type=type(error).__name__, report_written=False,\n                          eligible_for_future_fold_local_readout_fitting=False)\n    return report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--input-root", default="/kaggle/input")\n    parser.add_argument("--output-root", default="/kaggle/working/stage13a_admission")\n    arguments = parser.parse_args()\n    report = audit_inputs(arguments.input_root, arguments.output_root)\n    print("STAGE13A_INPUT_ADMISSION_JSON_BEGIN", flush=True)\n    print(json.dumps(report, sort_keys=True), flush=True)\n    print("STAGE13A_INPUT_ADMISSION_JSON_END", flush=True)\n    return 0 if report["status"] == "PASS_FROZEN_BASE_FEATURE_ADMISSION_ONLY" else 2\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
_runner_source = '"""Fit one fixed GLM readout per canonical prompt fold; Kaggle CPU only."""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\nimport warnings\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn import __version__ as sklearn_version\nfrom sklearn.exceptions import ConvergenceWarning\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.model_selection import GroupKFold\nfrom sklearn.preprocessing import StandardScaler\nfrom threadpoolctl import threadpool_info, threadpool_limits\n\nimport stage13a_cache_admission as admission\n\n\nCOMPLETE = "COMPLETE_GLM_PLAIN_GROUPED_DEVELOPMENT_REFERENCE"\nSCOPE = "glm_only_prompt_heldout_development"\nREQUIRED_SKLEARN = "1.6.1"\nTHREADS = 2\nHEAD = dict(C=0.0025, solver="lbfgs", max_iter=5000, tol=1e-4,\n            fit_intercept=True, random_state=0)\nPROBABILITY_TOL = 1e-12\nAUDIT_CHECKS = ["fresh_output_directory", "unique_required_inputs", "exact_input_file_hashes",\n                "official_train_axis_contract", "numeric_unicode_npz_headers",\n                "glm_values_alignment_and_finiteness", "canonical_future_prompt_fold_partitions"]\n\n\ndef validate_probabilities(values, rows, classes):\n    admission.check(values.shape == (rows, classes) and values.dtype == np.dtype("float64"),\n                    "probability_shape_or_dtype")\n    admission.check(np.isfinite(values).all() and (values > 0).all() and (values <= 1).all(),\n                    "invalid_probability_values")\n    error = float(np.max(np.abs(values.sum(axis=1) - 1.0)))\n    admission.check(error <= PROBABILITY_TOL, "probability_rows_not_normalized")\n    return error\n\n\ndef metrics(targets, probabilities, classes):\n    predicted = probabilities.argmax(axis=1)\n    counts = np.bincount(targets * classes + predicted, minlength=classes**2).reshape(classes, classes)\n    supports = counts.sum(axis=1)\n    admission.check((supports > 0).all() and counts.sum() == len(targets), "confusion_count_reconciliation")\n    recall = counts.diagonal() / supports\n    return float(recall.mean()), counts, supports, recall\n\n\ndef fit_grouped(features, targets, groups, folds, classes, n_folds, *, on_fold=None):\n    """Use the same feature/estimator contract in the synthetic and production paths."""\n    rows, class_count = len(targets), len(classes)\n    admission.check(features.ndim == 2 and len(features) == rows and features.dtype == np.float64\n                    and features.flags.c_contiguous and np.isfinite(features).all(), "invalid_readout_features")\n    admission.check(targets.shape == groups.shape == folds.shape == (rows,)\n                    and targets.dtype.kind in "iu"\n                    and np.array_equal(np.unique(targets), np.arange(class_count)), "invalid_readout_axes")\n    _, summaries = admission.validate_fold_partition(groups, folds, n_folds)\n    # Check all folds before the first fit, so partial coverage cannot yield a misleading score.\n    for number in range(1, n_folds + 1):\n        mask = folds == number\n        for subset in (mask, ~mask):\n            admission.check(np.array_equal(np.unique(targets[subset]), np.arange(class_count)),\n                            "fold_missing_class")\n    probabilities = np.full((rows, class_count), np.nan, dtype=np.float64)\n    filled = np.zeros(rows, dtype=np.int8)\n    with threadpool_limits(limits=THREADS):\n        pools = [{"api": item["user_api"], "threads": item["num_threads"]} for item in threadpool_info()]\n        admission.check(all(0 < p["threads"] <= THREADS for p in pools), "thread_limit_not_applied")\n        for summary in summaries:\n            started = time.monotonic()\n            mask = folds == summary["fold"]\n            scaler = StandardScaler().fit(features[~mask])\n            with warnings.catch_warnings():\n                warnings.simplefilter("error", ConvergenceWarning)\n                classifier = LogisticRegression(**HEAD).fit(scaler.transform(features[~mask]), targets[~mask])\n            admission.check(np.array_equal(classifier.classes_, np.arange(class_count)), "readout_class_order")\n            iterations = [int(i) for i in classifier.n_iter_]\n            admission.check(len(iterations) == 1 and 0 < iterations[0] < HEAD["max_iter"], "readout_iteration_limit")\n            values = classifier.predict_proba(scaler.transform(features[mask]))\n            validate_probabilities(values, int(mask.sum()), class_count)\n            admission.check((filled[mask] == 0).all(), "duplicate_heldout_predictions")\n            probabilities[mask], filled[mask] = values, 1\n            score, _, support, _ = metrics(targets[mask], values, class_count)\n            summary.update(balanced_accuracy=score, iterations=iterations[0],\n                           seconds=round(time.monotonic() - started, 6), valid_class_supports=support.tolist())\n            if on_fold is not None:\n                on_fold(dict(summary))\n    admission.check((filled == 1).all(), "incomplete_heldout_predictions")\n    max_error = validate_probabilities(probabilities, rows, class_count)\n    return probabilities, summaries, pools, max_error\n\n\ndef save_private_cache(path, *, ids, groups, targets, folds, classes, probabilities):\n    arrays = dict(ids=ids, prompt_ids=groups, targets=targets, folds=folds,\n                  classes=np.asarray(classes, dtype=str), probabilities=probabilities)\n    admission.check(all(not value.dtype.hasobject for value in arrays.values()), "cache_object_dtype")\n    tmp = path.with_name(".heldout_probabilities.tmp")\n    try:\n        with tmp.open("xb") as stream:\n            np.savez(stream, **arrays)\n        with np.load(tmp, allow_pickle=False) as saved:\n            admission.check(set(saved.files) == set(arrays), "cache_readback_schema")\n            for key, value in arrays.items():\n                readback = saved[key]\n                admission.check(readback.dtype == value.dtype and np.array_equal(readback, value),\n                                "cache_readback_values")\n        digest = admission.sha256(tmp)\n        tmp.replace(path)\n        return digest\n    finally:\n        tmp.unlink(missing_ok=True)\n\n\ndef run(input_root, output_root, *, on_fold=None):\n    started = time.monotonic()\n    root, out = Path(input_root), Path(output_root)\n    report = dict(schema_version=1, status="FAILED_GLM_READOUT", scope=SCOPE, folds=[],\n                  fit_attempted=False, fit_performed=False, heldout_predictions_produced=False, balancing_performed=False,\n                  official_score_estimated=False, full_s07_oof_admitted=False,\n                  stage13b_batch_coupled_comparison_admitted=False, heldout_cache_readback_passed=False,\n                  raw_rows_or_predictions_exported=False, official_submission_started=False,\n                  model_weights_saved=False, expected_input_versions={"competition": 170484, "own_glm_output": 359405443},\n                  estimator=HEAD.copy(), classifier_layers=[8, 3], thread_limit=THREADS)\n    write_report = False\n    try:\n        admission.check(not out.exists() or (out.is_dir() and not any(out.iterdir())), "fresh_readout_output_directory")\n        out.mkdir(parents=True, exist_ok=True)\n        write_report = True\n        audited = admission.audit_inputs(root, out / "input_admission")\n        report["runtime_versions"] = audited["runtime_versions"]\n        report["admission_checks_passed"] = audited["checks_passed_count"]\n        report["admission_status"] = audited["status"]\n        if audited["status"] != "PASS_FROZEN_BASE_FEATURE_ADMISSION_ONLY":\n            report["admission_failure_code"] = audited.get("failure_code")\n        admission.check(audited["status"] == "PASS_FROZEN_BASE_FEATURE_ADMISSION_ONLY"\n                        and audited["contract_is_production"] is True and audited["report_written"] is True\n                        and audited["checks_passed"] == AUDIT_CHECKS, "real_input_admission_failed")\n        admission.check(sklearn_version == REQUIRED_SKLEARN, "scikit_learn_version_mismatch")\n        report.update(input_files=audited["files"], id_axis_sha256=audited["id_axis_sha256"],\n                      label_axis_sha256=audited["label_axis_sha256"],\n                      canonical_prompt_fold_mapping_sha256=audited["canonical_prompt_fold_mapping_sha256"])\n        train_path = admission.unique_file(root, "train.csv")\n        cache_path = admission.unique_file(root, "glm_features.npz")\n        admission.check(admission.sha256(train_path) == admission.TRAIN_SHA256\n                        and admission.sha256(cache_path) == admission.GLM_SHA256, "post_admission_input_hashes")\n        train = pd.read_csv(train_path, dtype=str, usecols=["id", "prompt_id", "model"])\n        ids, groups = train.id.to_numpy(dtype=str), train.prompt_id.to_numpy(dtype=str)\n        classes = admission.CLASSES\n        targets = train.model.map({name: i for i, name in enumerate(classes)}).to_numpy(dtype=np.int64)\n        with np.load(cache_path, allow_pickle=False) as cache:\n            admission.check(np.array_equal(cache["ids"], ids) and np.array_equal(cache["layers"], [8, 3, 24]),\n                            "readout_feature_axis_binding")\n            values = cache["features"]\n            features = np.empty((len(ids), 8192), dtype=np.float64, order="C")\n            features[:, :4096], features[:, 4096:] = values[:, 0, :], values[:, 1, :]\n            del values\n        folds = np.zeros(len(ids), dtype=np.int64)\n        for number, (_, valid) in enumerate(GroupKFold(5).split(features, targets, groups), 1):\n            admission.check((folds[valid] == 0).all(), "overlapping_outer_fold_indices")\n            folds[valid] = number\n        mapping, summaries = admission.validate_fold_partition(groups, folds, 5)\n        admission.check(admission.json_hash(mapping) == admission.FOLD_MAP_SHA256\n                        and summaries == audited["fold_partitions"], "readout_fold_map_mismatch")\n        admission.check(admission.json_hash(ids.tolist()) == audited["id_axis_sha256"], "readout_id_axis_hash")\n        report.update(rows=len(ids), prompts=len(mapping), classes=list(classes), fit_attempted=True,\n                      feature_shape=list(features.shape))\n\n        def record_fold(summary):\n            report["folds"].append(summary)\n            report["fit_performed"] = True\n            if on_fold is not None:\n                on_fold(summary)\n\n        probs, summaries, pools, error = fit_grouped(features, targets, groups, folds, classes, 5, on_fold=record_fold)\n        ba, counts, supports, recalls = metrics(targets, probs, len(classes))\n        admission.check(supports.tolist() == [audited["class_supports"][c] for c in classes], "class_support_mismatch")\n        admission.check(len(summaries) == 5 and sum(f["valid_rows"] for f in summaries) == len(ids),\n                        "heldout_fold_count_mismatch")\n        cache_hash = save_private_cache(out / "heldout_probabilities.npz", ids=ids, groups=groups,\n                                        targets=targets, folds=folds, classes=classes, probabilities=probs)\n        report.update(status=COMPLETE, fit_performed=True, heldout_predictions_produced=True,\n                      heldout_cache_readback_passed=True, heldout_cache_sha256=cache_hash,\n                      probability_shape=list(probs.shape), probability_dtype=str(probs.dtype),\n                      probability_values_sha256=hashlib.sha256(probs.astype("<f8", copy=False).tobytes()).hexdigest(),\n                      probability_max_row_sum_error=error, pooled_balanced_accuracy=ba,\n                      confusion_counts=counts.tolist(), class_supports=supports.tolist(), class_recalls=recalls.tolist(),\n                      convergence_warnings=0, observed_thread_pools=pools)\n    except admission.AdmissionFailure as error:\n        report.update(failure_code=str(error), failure_type="AdmissionFailure")\n    except ConvergenceWarning:\n        report.update(failure_code="readout_convergence_warning", failure_type="ConvergenceWarning")\n    except Exception as error:\n        # Exception strings may contain paths or row values; publish only the type.\n        report.update(failure_code="readout_runtime_failure", failure_type=type(error).__name__)\n    report["seconds"] = round(time.monotonic() - started, 6)\n    if write_report:\n        try:\n            tmp = out / ".readout_report.tmp"\n            tmp.write_text(json.dumps(report, sort_keys=True, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n            tmp.replace(out / "readout_report.json")\n        except Exception as error:\n            report.update(status="FAILED_GLM_READOUT", failure_code="readout_report_write_failed",\n                          failure_type=type(error).__name__)\n    return report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--input-root", default="/kaggle/input")\n    parser.add_argument("--output-root", default="/kaggle/working/stage13_glm_readout")\n    args = parser.parse_args()\n\n    def progress(fold):\n        print("STAGE13_GLM_FOLD", json.dumps(fold, sort_keys=True, allow_nan=False), flush=True)\n\n    report = run(args.input_root, args.output_root, on_fold=progress)\n    print("STAGE13_GLM_READOUT_JSON_BEGIN", flush=True)\n    print(json.dumps(report, sort_keys=True, allow_nan=False), flush=True)\n    print("STAGE13_GLM_READOUT_JSON_END", flush=True)\n    return 0 if report["status"] == COMPLETE else 2\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
_admission_sha256 = '4b30488d126bc6213d8e1d128d50d7f0641b33e90b13192af55abe413ae49629'
_runner_sha256 = 'a4fe8b337162fbbbf409e3bb8bc0fcfb84711bfbef52bb61bdc5ca69b8329596'
_source_dir = Path('/kaggle/working/_stage13_glm_readout_source')
_runner_path = _source_dir / 'stage13_glm_readout.py'
_admission_path = _source_dir / 'stage13a_cache_admission.py'
for _source, _expected in [(_admission_source, _admission_sha256), (_runner_source, _runner_sha256)]:
    if hashlib.sha256(_source.encode('utf-8')).hexdigest() != _expected:
        raise RuntimeError('Stage13 embedded source hash mismatch')
_source_dir.mkdir(parents=True, exist_ok=True)
for _path, _source, _expected in [(_admission_path, _admission_source, _admission_sha256), (_runner_path, _runner_source, _runner_sha256)]:
    _path.write_bytes(_source.encode('utf-8'))
    if hashlib.sha256(_path.read_bytes()).hexdigest() != _expected:
        raise RuntimeError('Stage13 written source hash mismatch')
print('STAGE13_GLM_READOUT_SOURCE_CHECKS_PASS')


## Fit and evaluate the frozen recipe

A failed input, fold or model check stops the run. The row-level cache must remain in private Kaggle output. Logs contain only aggregate evidence and digests.


In [ ]:
subprocess.run([sys.executable, str(_runner_path), '--input-root', '/kaggle/input', '--output-root', '/kaggle/working/stage13_glm_readout'], check=True, timeout=1800)


## Check the saved result

Verify the aggregate report, fold scope and private cache digest before accepting this development reference. No cache rows are displayed.


In [ ]:
_result_dir = Path('/kaggle/working/stage13_glm_readout')
_report = json.loads((_result_dir / 'readout_report.json').read_text(encoding='utf-8'))
def _require(condition, code):
    if not condition:
        raise RuntimeError('Stage13 saved report check failed: ' + code)
_require(_report.get('status') == 'COMPLETE_GLM_PLAIN_GROUPED_DEVELOPMENT_REFERENCE', 'status')
_require(_report.get('scope') == 'glm_only_prompt_heldout_development', 'scope')
_require((_report.get('rows'), _report.get('prompts'), _report.get('admission_checks_passed')) == (2699, 150, 7), 'admission_axes')
_require(_report.get('probability_shape') == [2699, 18], 'probability_shape')
for _key in ['fit_performed', 'heldout_predictions_produced', 'heldout_cache_readback_passed']:
    _require(_report.get(_key) is True, _key)
for _key in ['balancing_performed', 'official_score_estimated', 'full_s07_oof_admitted', 'stage13b_batch_coupled_comparison_admitted']:
    _require(_report.get(_key) is False, _key)
_folds = _report.get('folds')
_require(isinstance(_folds, list) and len(_folds) == 5, 'fold_count')
for _number, _fold in enumerate(_folds, 1):
    _valid_rows = 540 if _number <= 4 else 539
    _require(isinstance(_fold, dict) and _fold.get('fold') == _number, 'fold_order')
    _require((_fold.get('fit_rows'), _fold.get('valid_rows'), _fold.get('fit_prompts'), _fold.get('valid_prompts'), _fold.get('prompt_overlap')) == (2699 - _valid_rows, _valid_rows, 120, 30, 0), 'fold_partition')
_score = _report.get('pooled_balanced_accuracy')
_require(type(_score) in (float, int) and math.isfinite(_score) and 0 <= _score <= 1, 'development_score')
with (_result_dir / 'heldout_probabilities.npz').open('rb') as _stream:
    _actual_cache_sha256 = hashlib.file_digest(_stream, 'sha256').hexdigest()
_require(_actual_cache_sha256 == _report.get('heldout_cache_sha256'), 'cache_hash')
print('STAGE13_GLM_READOUT_SAVED_REPORT_READBACK_PASS', json.dumps({'rows': _report['rows'], 'folds': len(_folds), 'development_balanced_accuracy': _score, 'heldout_cache_sha256': _actual_cache_sha256}, sort_keys=True))
